In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix
from xgboost import XGBClassifier

In [2]:
# ==========================================
# LOAD DATA
# ==========================================
df = pd.read_csv("clean_zeek_format_dataset.csv")

features = [
    "duration", "orig_bytes", "resp_bytes",
    "orig_pkts", "resp_pkts", "id.resp_p",
    "total_bytes", "total_packets"
]

X = df[features]
y = df["Label"]

In [3]:
# ==========================================
# TRAIN TEST SPLIT
# ==========================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

In [4]:
# ==========================================
# FEATURE SCALING
# ==========================================
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [5]:
# ==========================================
# ISOLATION FOREST (UPDATED)
# ==========================================

from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# -------------------------------
# Train ONLY on NORMAL traffic
# -------------------------------
X_train_normal = X_train_scaled[y_train == 0]

# 🔥 IMPORTANT: lower contamination
iso_model = IsolationForest(
    contamination=0.01,   # ↓ reduce false positives
    random_state=42
)

iso_model.fit(X_train_normal)

# -------------------------------
# Use decision function (BETTER)
# -------------------------------
iso_scores = iso_model.decision_function(X_test_scaled)

# 🔥 Custom threshold (tunable)
threshold = -0.15
y_pred_iso = (iso_scores < threshold).astype(int)

# -------------------------------
# Evaluation
# -------------------------------
print("=== Isolation Forest (Updated) ===")
print(classification_report(y_test, y_pred_iso))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_iso))

# -------------------------------
# Save Model (IMPORTANT)
# -------------------------------
import joblib

joblib.dump({
    "model": iso_model
}, "models/isolation_forest_model.pkl")

=== Isolation Forest (Updated) ===


/home/kali/Documents/Intrusion-Detection-System-IDS-using-Neural-Networks/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/kali/Documents/Intrusion-Detection-System-IDS-using-Neural-Networks/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/kali/Documents/Intrusion-Detection-System-IDS-using-Neural-Networks/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 

              precision    recall  f1-score   support

           0       0.83      1.00      0.91    628946
           1       0.00      0.00      0.00    127763

    accuracy                           0.83    756709
   macro avg       0.42      0.50      0.45    756709
weighted avg       0.69      0.83      0.75    756709

Confusion Matrix:
[[628946      0]
 [127763      0]]


['models/isolation_forest_model.pkl']

In [6]:
# ==========================================
# XGBOOST MODEL (UPDATED)
# ==========================================

from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import joblib

# -------------------------------
# Handle class imbalance
# -------------------------------
scale_pos_weight = len(y_train[y_train == 0]) / len(y_train[y_train == 1])

# -------------------------------
# Model
# -------------------------------
xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,   # 🔥 IMPORTANT
    eval_metric='logloss',
    use_label_encoder=False
)

# -------------------------------
# Train
# -------------------------------
xgb_model.fit(X_train_scaled, y_train)

# -------------------------------
# Predict using probability
# -------------------------------
y_prob_xgb = xgb_model.predict_proba(X_test_scaled)[:, 1]

# Threshold (can tune later)
threshold = 0.5
y_pred_xgb = (y_prob_xgb > threshold).astype(int)

# -------------------------------
# Evaluation
# -------------------------------
print("=== XGBoost (Updated) ===")
print(classification_report(y_test, y_pred_xgb))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_xgb))

# -------------------------------
# Save Model
# -------------------------------
joblib.dump({
    "model": xgb_model,
    "scaler": scaler
}, "models/xgboost_model.pkl")

/home/kali/Documents/Intrusion-Detection-System-IDS-using-Neural-Networks/.venv/lib/python3.13/site-packages/xgboost/training.py:200: UserWarning: [17:11:47] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


=== XGBoost (Updated) ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    628946
           1       0.98      0.99      0.99    127763

    accuracy                           1.00    756709
   macro avg       0.99      0.99      0.99    756709
weighted avg       1.00      1.00      1.00    756709

Confusion Matrix:
[[626968   1978]
 [  1205 126558]]


['models/xgboost_model.pkl']

In [7]:
# ==========================================
# SAVE MODELS (VERY IMPORTANT)
# ==========================================

joblib.dump({
    "model": iso_model,
    "scaler": scaler
}, "isolation_forest_model.pkl")

joblib.dump({
    "model": xgb_model,
    "scaler": scaler
}, "xgboost_model.pkl")

print("✅ Models saved successfully!")

✅ Models saved successfully!
